In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from dotenv import load_dotenv
from pathlib import Path
from utils.utils import sliding_windows
import joblib
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import zscore

load_dotenv('../.env')

True

In [2]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

BASE_PATH = os.getenv("BASE_PATH")
PICKLE_PATH = BASE_PATH + os.getenv("PICKLE_PATH")

In [ ]:
eo_data = joblib.load(PICKLE_PATH + 'eo_crop.pkl')
ec_data = joblib.load(PICKLE_PATH + 'ec_crop.pkl')


def split_raw_by_time(raw_list, train_ratio=0.8, val_ratio=0.1):
    train_raw, val_raw, test_raw = [], [], []

    for raw in raw_list:
        sfreq = raw.info['sfreq']
        n_times = raw.n_times
        train_end = int(n_times * train_ratio)
        val_end = int(n_times * (train_ratio + val_ratio))

        train_raw.append(raw.copy().crop(tmin=0, tmax=(train_end - 1) / sfreq))
        val_raw.append(raw.copy().crop(tmin=train_end / sfreq, tmax=(val_end - 1) / sfreq))
        test_raw.append(raw.copy().crop(tmin=val_end / sfreq, tmax=(n_times - 1) / sfreq))

    return train_raw, val_raw, test_raw


eo_train_raw, eo_val_raw, eo_test_raw = split_raw_by_time(eo_data)
ec_train_raw, ec_val_raw, ec_test_raw = split_raw_by_time(ec_data)

X_eo_train, y_eo_train = sliding_windows(eo_train_raw)
X_eo_val, y_eo_val = sliding_windows(eo_val_raw)
X_eo_test, y_eo_test = sliding_windows(eo_test_raw)

X_ec_train, y_ec_train = sliding_windows(ec_train_raw)
X_ec_val, y_ec_val = sliding_windows(ec_val_raw)
X_ec_test, y_ec_test = sliding_windows(ec_test_raw)

X_eo_train = np.array(X_eo_train)
X_eo_val = np.array(X_eo_val)
X_eo_test = np.array(X_eo_test)
y_eo_train = np.array(y_eo_train)
y_eo_val = np.array(y_eo_val)
y_eo_test = np.array(y_eo_test)

X_ec_train = np.array(X_ec_train)
X_ec_val = np.array(X_ec_val)
X_ec_test = np.array(X_ec_test)
y_ec_train = np.array(y_ec_train)
y_ec_val = np.array(y_ec_val)
y_ec_test = np.array(y_ec_test)

print("EO train/val/test:", X_eo_train.shape, X_eo_val.shape, X_eo_test.shape)
print("EC train/val/test:", X_ec_train.shape, X_ec_val.shape, X_ec_test.shape)

EO train/val/test: (5232, 64, 160) (654, 64, 160) (654, 64, 160)
EC train/val/test: (5232, 64, 160) (654, 64, 160) (654, 64, 160)


In [4]:
X_eo_train = zscore(X_eo_train, axis=2)
X_eo_val = zscore(X_eo_val, axis=2)
X_eo_test = zscore(X_eo_test, axis=2)

X_ec_train = zscore(X_ec_train, axis=2)
X_ec_val = zscore(X_ec_val, axis=2)
X_ec_test = zscore(X_ec_test, axis=2)

In [5]:
PREPROCESSED_PATH = BASE_PATH + os.getenv("PREPROCESSED_PATH")
PREPROCESSED_DIR = Path(PREPROCESSED_PATH)
PREPROCESSED_DIR.mkdir(exist_ok=True)

suffix = f"{os.getenv('WINDOW_SIZE').replace('.', '')}_{os.getenv('STRIDE').replace('.', '')}"

np.save(PREPROCESSED_DIR / f'X_eo_train_{suffix}.npy', X_eo_train)
np.save(PREPROCESSED_DIR / f'X_eo_val_{suffix}.npy', X_eo_val)
np.save(PREPROCESSED_DIR / f'X_eo_test_{suffix}.npy', X_eo_test)
np.save(PREPROCESSED_DIR / f'y_eo_train_{suffix}.npy', y_eo_train)
np.save(PREPROCESSED_DIR / f'y_eo_val_{suffix}.npy', y_eo_val)
np.save(PREPROCESSED_DIR / f'y_eo_test_{suffix}.npy', y_eo_test)

np.save(PREPROCESSED_DIR / f'X_ec_train_{suffix}.npy', X_ec_train)
np.save(PREPROCESSED_DIR / f'X_ec_val_{suffix}.npy', X_ec_val)
np.save(PREPROCESSED_DIR / f'X_ec_test_{suffix}.npy', X_ec_test)
np.save(PREPROCESSED_DIR / f'y_ec_train_{suffix}.npy', y_ec_train)
np.save(PREPROCESSED_DIR / f'y_ec_val_{suffix}.npy', y_ec_val)
np.save(PREPROCESSED_DIR / f'y_ec_test_{suffix}.npy', y_ec_test)

In [6]:
print("Proses Selesai! Data siap masuk Embedding Model.")
print("Verifikasi Mean (harus ~0):", np.mean(X_eo_train[0, 0, :]))
print("Verifikasi Std (harus 1):", np.std(X_eo_train[0, 0, :]))

Proses Selesai! Data siap masuk Embedding Model.
Verifikasi Mean (harus ~0): 6.245004513516506e-17
Verifikasi Std (harus 1): 0.9999999999999999
